# Amharic LLM LoRA Fine-Tuning

A reproducible notebook for parameter-efficient fine-tuning of a causal language model on an Amharic sentence corpus using **LoRA/PEFT**.

- Dataset: `a3xrfgb/amharic-sentences-corpus`
- Default base model: `Qwen/Qwen3-0.6B`
- Objective: causal language-model training on raw Amharic sentences
- Default method: LoRA
- Optional: 4-bit QLoRA-style model loading


## 1. References

- Dataset: https://huggingface.co/datasets/a3xrfgb/amharic-sentences-corpus
- Qwen3-0.6B: https://huggingface.co/Qwen/Qwen3-0.6B
- Qwen3-0.6B-Base: https://huggingface.co/Qwen/Qwen3-0.6B-Base
- TRL SFTTrainer: https://huggingface.co/docs/trl/sft_trainer
- PEFT/LoRA integration: https://huggingface.co/docs/trl/peft_integration


In [ ]:
# Install current Hugging Face training dependencies.
# Restart the kernel/runtime after this cell if your environment asks you to.

%pip install -U "transformers>=4.51.0" "datasets>=3.0.0" "trl>=0.26.0" "peft>=0.14.0" "accelerate>=1.2.0" "huggingface_hub>=0.30.0" sentencepiece safetensors

# Optional for 4-bit loading. If you do not use 4-bit mode, this can be skipped.
%pip install -U bitsandbytes

In [ ]:
import os
import re
import math
import random
import json
import platform
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
print('Transformers / TRL / PEFT imported successfully.')

In [ ]:
# =========================
# Main configuration
# =========================

DATASET_NAME = 'a3xrfgb/amharic-sentences-corpus'
DATASET_SPLIT = 'train'

# Recommended starter model. For more capacity, try Qwen/Qwen3-1.7B or another
# causal model whose tokenizer has good Amharic coverage.
MODEL_NAME = 'Qwen/Qwen3-0.6B'

OUTPUT_DIR = './amharic-qwen3-0.6b-lora'
ADAPTER_DIR = './amharic-qwen3-0.6b-lora/adapter'

# Dataset controls.
# None = use the full corpus. For a quick smoke test, use e.g. 10_000.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = 5_000
VALIDATION_RATIO = 0.01

# Sequence/training settings.
MAX_LENGTH = 1024
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# Batch size is intentionally conservative. Increase if GPU memory allows.
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

# LoRA settings.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

# Hardware / precision.
USE_4BIT = False  # Set True for 4-bit loading on supported CUDA environments.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

# Data processing. None lets TRL choose; lower values may help on smaller CPUs.
DATASET_NUM_PROC = None

SEED = 42
set_seed(SEED)

print('MODEL_NAME =', MODEL_NAME)
print('USE_4BIT =', USE_4BIT)
print('BF16 =', USE_BF16, '| FP16 =', USE_FP16)
print('MAX_TRAIN_SAMPLES =', MAX_TRAIN_SAMPLES)

## 2. Load and inspect the Amharic corpus

The dataset is hosted on the Hugging Face Hub and can be loaded with `load_dataset()`. The notebook does not assume the text column name in advance; it detects a suitable string column automatically and then normalizes it to `text`.

In [ ]:
raw = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

print(raw)
print('Columns:', raw.column_names)
print('Rows:', len(raw))
print('\nFirst row:')
print(raw[0])

In [ ]:
def find_text_column(ds):
    """Find the most plausible string/text column automatically."""
    preferred = ['text', 'sentence', 'content', 'text_am', 'amharic', 'sentence_text']
    for col in preferred:
        if col in ds.column_names:
            feature = ds.features[col]
            if str(feature).lower().find('string') >= 0:
                return col
    string_columns = []
    for col, feature in ds.features.items():
        if str(feature).lower().find('string') >= 0:
            string_columns.append(col)
    if not string_columns:
        raise ValueError(f'No string column found. Columns: {ds.column_names}')
    return string_columns[0]

TEXT_COLUMN = find_text_column(raw)
print('Detected text column:', TEXT_COLUMN)

for i in range(min(5, len(raw))):
    print(f'[{i}] {raw[i][TEXT_COLUMN]}')

In [ ]:
# Basic text statistics before cleaning.
sample_for_stats = raw.select(range(min(20_000, len(raw))))
lengths = [len(str(x or '')) for x in sample_for_stats[TEXT_COLUMN]]

print('Character-length statistics on first', len(lengths), 'rows:')
print(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# Count empty/very short examples on the sample.
empty_count = sum(len(str(x or '').strip()) == 0 for x in sample_for_stats[TEXT_COLUMN])
short_count = sum(len(str(x or '').strip()) < 5 for x in sample_for_stats[TEXT_COLUMN])
print('Empty:', empty_count, '| <5 chars:', short_count)

## 3. Clean, deduplicate, and split the data

The cleaning below is deliberately conservative. It normalizes whitespace, removes empty rows, and removes exact duplicate sentences. It does **not** attempt aggressive Amharic orthographic normalization, because that can unintentionally alter useful linguistic information.

If you want to reproduce the corpus exactly without deduplication, set `DEDUPLICATE = False`.

In [ ]:
DEDUPLICATE = True

def clean_text(example):
    text = str(example[TEXT_COLUMN] or '')
    text = text.replace('\u0000', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return {'text': text}

dataset = raw.map(
    clean_text,
    num_proc=DATASET_NUM_PROC,
    desc='Normalizing Amharic text',
)

dataset = dataset.filter(
    lambda x: 5 <= len(x['text']) <= 20_000,
    num_proc=DATASET_NUM_PROC,
    desc='Removing empty/invalid examples',
)

if DEDUPLICATE:
    before = len(dataset)
    dataset = dataset.drop_duplicates('text')
    print('Removed exact duplicates:', before - len(dataset))

print('Clean rows:', len(dataset))
print(dataset[0])

In [ ]:
# Create a reproducible train/evaluation split.
splits = dataset.train_test_split(test_size=VALIDATION_RATIO, seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

if MAX_TRAIN_SAMPLES is not None:
    n = min(MAX_TRAIN_SAMPLES, len(train_ds))
    train_ds = train_ds.shuffle(seed=SEED).select(range(n))

if MAX_EVAL_SAMPLES is not None:
    n = min(MAX_EVAL_SAMPLES, len(eval_ds))
    eval_ds = eval_ds.select(range(n))

print('Train rows:', len(train_ds))
print('Eval rows:', len(eval_ds))
print('Example:', train_ds[0]['text'])

## 4. Load tokenizer and inspect Amharic tokenization

Tokenizer efficiency matters for Amharic because it directly affects sequence length, GPU memory, and the amount of language seen per training step.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Tokenizer:', tokenizer.__class__.__name__)
print('Vocab size:', tokenizer.vocab_size)
print('EOS:', repr(tokenizer.eos_token))
print('PAD:', repr(tokenizer.pad_token))

examples = train_ds['text'][:5]
for text in examples:
    ids = tokenizer(text, add_special_tokens=False)['input_ids']
    print('\nTEXT:', text)
    print('Characters:', len(text), 'Tokens:', len(ids), 'Chars/token:', round(len(text)/max(1,len(ids)), 2))

## 5. Configure LoRA

LoRA freezes the base model and trains low-rank adapter matrices. TRL's current SFTTrainer integrates directly with PEFT. A learning rate around `1e-4` to `2e-4` is a common starting point for LoRA; this notebook uses `2e-4` as a practical starting value.

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)

print(lora_config)

## 6. Optional 4-bit loading

Leave `USE_4BIT=False` for standard LoRA. Set it to `True` if you need to reduce GPU memory usage and your CUDA environment supports `bitsandbytes`. This turns the setup into a QLoRA-style training configuration while still learning LoRA adapters.

In [ ]:
quantization_config = None
model_kwargs = {}

if USE_4BIT:
    if not torch.cuda.is_available():
        raise RuntimeError('USE_4BIT=True requires a CUDA GPU with bitsandbytes support.')
    compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model_kwargs['quantization_config'] = quantization_config
    model_kwargs['device_map'] = 'auto'
else:
    if USE_BF16:
        model_kwargs['torch_dtype'] = torch.bfloat16
    elif USE_FP16:
        model_kwargs['torch_dtype'] = torch.float16

print('Model kwargs:', model_kwargs)

In [ ]:
# Load the base model.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False

# This is important when using gradient checkpointing.
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

print('Loaded:', MODEL_NAME)
print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total

# We intentionally let SFTTrainer attach the LoRA adapter through peft_config.
# This cell reports the base-model parameter count before adapter injection.
trainable, total, pct = count_trainable_parameters(model)
print(f'Before LoRA: trainable={trainable:,} / total={total:,} ({pct:.4f}%)')

## 7. Configure SFTTrainer

The corpus is plain text, so `dataset_text_field='text'` is used. `packing=True` allows multiple short sentences to share a training sequence, which can improve utilization for a sentence corpus.

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=False,
    bf16=USE_BF16,
    fp16=USE_FP16,
    tf32=torch.cuda.is_available(),
    optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    lr_scheduler_type='cosine',
    report_to='none',
    seed=SEED,
    dataset_text_field='text',
    max_length=MAX_LENGTH,
    packing=True,
    dataset_num_proc=DATASET_NUM_PROC,
    model_init_kwargs=model_kwargs,
)

print(training_args)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print('Trainer created.')
print('Trainable parameter summary:')
trainer.model.print_trainable_parameters()

## 8. Estimate training size

The calculation below gives a rough estimate of optimizer steps. Actual runtime depends heavily on GPU, sequence lengths, packing efficiency, and dataloader throughput.

In [ ]:
effective_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
estimated_steps_per_epoch = math.ceil(len(train_ds) / effective_batch)
estimated_total_steps = estimated_steps_per_epoch * NUM_EPOCHS
print('Approx. examples / optimizer step:', effective_batch)
print('Approx. optimizer steps / epoch:', estimated_steps_per_epoch)
print('Approx. total optimizer steps:', estimated_total_steps)
print('\nFor a quick smoke test, set MAX_TRAIN_SAMPLES=1_000 to 10_000.')

## 9. Train the LoRA adapter

For the full 1.6M-row corpus, this may take substantial time. The notebook saves checkpoints according to `save_steps` and the final adapter to `ADAPTER_DIR`.

In [ ]:
train_result = trainer.train()
print(train_result)

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved LoRA adapter to:', ADAPTER_DIR)

## 10. Evaluate the fine-tuned model

Perplexity is computed from evaluation loss as `exp(loss)`. For language modeling, lower perplexity generally means the model assigns higher probability to the held-out text. Compare the fine-tuned model against the base model on the **same held-out set** if you want a meaningful before/after measurement.

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

if 'eval_loss' in eval_metrics:
    try:
        perplexity = math.exp(eval_metrics['eval_loss'])
        print('Perplexity:', perplexity)
    except OverflowError:
        print('Perplexity overflowed because eval_loss is very large.')

## 11. Generate Amharic text with the trained adapter

This uses the trained `PeftModel` already attached to the trainer. Qwen3 can operate in non-thinking mode for ordinary text generation; for this corpus-completion experiment, `enable_thinking=False` keeps generation focused on direct text continuation.

In [ ]:
prompts = [
    'ኢትዮጵያ በአፍሪካ',
    'ዛሬ በአዲስ አበባ',
    'የአማርኛ ቋንቋ',
]

generation_model = trainer.model
generation_model.eval()

for prompt in prompts:
    messages = [{'role': 'user', 'content': prompt}]
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
            enable_thinking=False,
        )
    except TypeError:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )

    device = next(generation_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs['input_ids'].shape[-1]:]
    print('\nPROMPT:', prompt)
    print('OUTPUT:', tokenizer.decode(generated, skip_special_tokens=True))

## 12. Reload the adapter from disk

The adapter directory contains the small LoRA weights rather than a full copy of the base model. You can later reload the base model plus adapter for inference.

In [ ]:
from peft import AutoPeftModelForCausalLM

reload_kwargs = {}
if USE_4BIT:
    reload_kwargs['quantization_config'] = quantization_config
    reload_kwargs['device_map'] = 'auto'
elif USE_BF16:
    reload_kwargs['torch_dtype'] = torch.bfloat16
elif USE_FP16:
    reload_kwargs['torch_dtype'] = torch.float16

reloaded_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    **reload_kwargs,
)
reloaded_model.eval()
print('Reloaded adapter successfully.')

## 13. Optional: merge LoRA into the base model

If you need a standalone model directory without PEFT adapter separation, you can merge the adapter into the base weights. This is optional and requires enough RAM/VRAM to hold the merged model.

In [ ]:
MERGED_DIR = './amharic-qwen3-0.6b-merged'
MERGE_ADAPTER = False

if MERGE_ADAPTER:
    merged_model = reloaded_model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print('Merged model saved to:', MERGED_DIR)
else:
    print('Skipping merge. Set MERGE_ADAPTER=True to create a standalone merged model.')

## 14. Optional: upload the adapter to Hugging Face Hub

First authenticate with `huggingface-cli login` or `notebook_login()`. Then set your repository ID and run the upload cell.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# HUB_REPO_ID = 'YOUR_USERNAME/amharic-qwen3-0.6b-lora'
# trainer.push_to_hub(HUB_REPO_ID)
# tokenizer.push_to_hub(HUB_REPO_ID)

print('Hub upload is disabled by default. Uncomment and configure HUB_REPO_ID when ready.')

## 15. Practical tuning guidance

### GPU memory
- If you get CUDA OOM, reduce `PER_DEVICE_TRAIN_BATCH_SIZE` to `1`.
- Keep or increase `GRADIENT_ACCUMULATION_STEPS` to preserve the effective batch size.
- Reduce `MAX_LENGTH` from `1024` to `512`.
- Set `USE_4BIT=True` for a lower-memory QLoRA-style setup.

### Model size
- `Qwen/Qwen3-0.6B` is a good smoke-test model.
- For a stronger experiment, try `Qwen/Qwen3-1.7B` if your hardware can handle it.
- For still larger models, QLoRA, gradient checkpointing, and multiple GPUs become increasingly useful.

### Dataset size
The corpus is large enough that a full run can be expensive. Start with `MAX_TRAIN_SAMPLES=10_000` to verify that the entire notebook works, then set it back to `None` for the full corpus.

### Language-model objective
Because the source dataset consists of sentences rather than instruction/response pairs, this notebook teaches the model Amharic text continuation. It does **not** by itself create an Amharic chat assistant. For instruction tuning, you would need an instruction dataset such as question/answer or prompt/completion examples.

### Evaluation
Perplexity alone is not enough to measure practical Amharic quality. For a serious experiment, build a held-out Amharic benchmark covering generation, comprehension, summarization, translation, spelling/grammar, and domain-specific tasks relevant to your intended application.

In [ ]:
# Final configuration summary for reproducibility.
config = {
    'dataset': DATASET_NAME,
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'lora_target_modules': LORA_TARGET_MODULES,
    'use_4bit': USE_4BIT,
    'bf16': USE_BF16,
    'fp16': USE_FP16,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_eval_samples': MAX_EVAL_SAMPLES,
    'seed': SEED,
}
print(json.dumps(config, indent=2, ensure_ascii=False))